In [ ]:
# %matplotlib inline
%time from hikyuu.interactive import *

# 1 Plot the indicators

In [ ]:
s = sm['sz000001']
k = s.get_kdata(Query(-200))


# Extract the close price indicator of the K-line; generally the calculation parameters of an indicator can only be of the indicator type, so the K-line data must first be converted to the indicator type
c = CLOSE(k)

# Calculate the EMA indicator of the close price
a = EMA(c)

# Draw the indicator
c.plot(legend_on=True)
a.plot(new=False, legend_on=True)

# Draw the bar chart
a.bar()

# Adjust the bar chart to make it more beautiful
PRICELIST([x-9 for x in a]).bar()

# 2 Indicators (Indicator)

In Hikyuu, the Indicator instance is the main data structure for calculation. Generally the calculation parameter of ind (if not specified, ind represents an Indicator instance) is another ind, e.g. EMA(x), where x should be an Indicator instance. It can be simply understood as similar to numpy.array.

## 2.1 Special indicators

There is a kind of special indicators used to convert the K-line data or an ordinary array to an ind, so that other inds can calculate with it. For example, KDATA converts KData to an ind. Others include: OPEN, HIGH, LOW, CLOSE, AMO (amount), VOL (volume), KDATA_PART.

In [ ]:
print("k is a instance of KData:\n", k)
print("--------------------------\n")

kind = KDATA(k)
print("kind is a instance of Indicator:\n", kind)

In [ ]:
# Get the number of the result sets of ind, e.g. MACD usually returns 3 result sets
r = kind.get_result_num()
print("result_num: ", r)

# Get the first result set
x = kind.get_result(0)
print(x)

In [ ]:
# The following are equivalent
c1 = CLOSE(k)
c2 = KDATA_PART(k, 'CLOSE')

Another commonly used special indicator PRICELIST wraps the Python list-like object into an ind.

In [ ]:
x = PRICELIST([i for i in range(100)])
print(len(x), x)

## 2.2 The properties and parameters of an Indicator

Every indicator function, such as EMA and HHV, generates an ind object after being called; the object itself can be called again to generate a new ind. Whether the indicator function or the ind object generates an ind, it is calculated immediately.

In [ ]:
e1 = EMA(CLOSE(k), n=5)
e2 = e1(CLOSE(k))
e3 = e2
print(e1 == e2)

Besides specifying the parameters in the indicator function, you can use the getParam and setParam methods to get and modify the parameters of the ind object. After modifying the parameters, the ind itself does not change; you need to call it to generate a new ind, and the new ind is the result calculated with the new parameters.

In [ ]:
e = EMA(c)
print(e)
print(e.get_param('n'))
e.plot(legend_on=True)

e.set_param('n', 30)
e = e(c)
e.plot(new=False, legend_on=True)

View the ind parameters. The ind parameters support:

- i : int
- s : str
- b : bool
- d : float

In [ ]:
# The EMA indicator has a parameter "n"; the type "i" means an integer
print(EMA())

## 2.3 TA-Lib wrapped indicators

TA-Lib is wrapped in the interactive tools with the naming convention TA_FUNC name. Among them, the lookback property of the ta-lib indicator is replaced by the discard property.

In [ ]:
x = TA_SMA(CLOSE(k))
print(x)
x.plot()

print(x.discard)

In [ ]:
query = Query(-200)
k1 = sm['sh000001'].get_kdata(query)
k2 = sm['sz000001'].get_kdata(query)

cr = TA_CORREL(CLOSE(k1), CLOSE(k2))
cr.plot()

## 2.4 Dynamic indicator parameters

In the securities market software such as the channel signal, the window parameters in the technical indicators usually support both integers and indicators, e.g.:

```
T1:=HHVBARS(H,120); {the number of the days from the highest point within 120 days to today}
L120:=LLV(L,T1+1); {the lowest point of this range from the highest point within 120 days to now}
```

***Since version 1.2.3, Hikyuu also starts to support using an indicator as the window parameter***

In [ ]:
h = HIGH(k)
l = LOW(k)
T1 = HHVBARS(h, 120)
L120 = LLV(l, T1+1)
L120.plot()

**Notes**

Since it is impossible to distinguish whether ind is an indicator parameter or the output data to be calculated in the Indicator(ind) form, if you want ind to be a parameter, you need to specify it explicitly with IndParam, e.g. EMA(IndParam(ind)).

The best way is to specify the parameter name to make it clear that a parameter is used:

```
x = EMA(c)  # use the close price as the calculation input
y = EMA(IndParam(c)) # use the close price as the n parameter
z = EMA(n=c) # use the close price as the parameter n
```

In [ ]:
# Or calculate in the prototype way by specifying the context
T1 = HHVBARS(H, 120)
L120 = LLV(L, T1+1)
L120.set_context(k)
L120.plot()